In [1]:
import torch
import cv2
import numpy as np
import pathlib
import mediapipe as mp
import logging
import math

# Windows‐fix cho torch.hub
pathlib.PosixPath = pathlib.WindowsPath

In [2]:
logging.basicConfig(
    filename='action_log.txt', filemode='a',
    format='%(asctime)s - %(message)s', level=logging.INFO
)
logging.getLogger().addHandler(logging.StreamHandler())

In [3]:
# --- 1) Load models
yolo_drink = torch.hub.load(
    'ultralytics/yolov5', 'custom',
    path='training/epochs_75/best.pt', force_reload=True
)
soft_model = torch.hub.load(
    'ultralytics/yolov5', 'custom',
    path='training/epochs_50/weights_1.pt', force_reload=True
)
# Hạ confidence threshold để bắt container tốt hơn
yolo_drink.conf = 0.3
soft_model.conf  = 0.5

Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to C:\Users\admin/.cache\torch\hub\master.zip
YOLOv5  2025-7-10 Python-3.10.16 torch-2.1.0+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)

Fusing layers... 
Model summary: 157 layers, 7023610 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to C:\Users\admin/.cache\torch\hub\master.zip
YOLOv5  2025-7-10 Python-3.10.16 torch-2.1.0+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)

Fusing layers... 
Model summary: 157 layers, 7085641 parameters, 0 gradients, 16.0 GFLOPs
Adding AutoShape... 


In [4]:
# --- 2) Mediapipe
mp_hands = mp.solutions.hands.Hands(max_num_hands=2, min_detection_confidence=0.5)
mp_face  = mp.solutions.face_mesh.FaceMesh(max_num_faces=1, min_detection_confidence=0.5)
SMOKE_DIST_THRESH = 0.25

In [5]:
# --- 3) Helpers drink
def segment_liquid_region(crop):
    h, w = crop.shape[:2]
    m = np.zeros((h, w), np.uint8)
    bg, fg = np.zeros((1,65),np.float64), np.zeros((1,65),np.float64)
    rect = (int(0.1*w), int(0.1*h), int(0.8*w), int(0.8*h))
    cv2.grabCut(crop, m, rect, bg, fg, 3, cv2.GC_INIT_WITH_RECT)
    m2 = np.where((m==2)|(m==0), 0, 1).astype('uint8')
    return crop * m2[:, :, None]

def classify_liquid(hsv):
    flat = hsv.reshape(-1,3)
    flat = flat[(flat.sum(1)>0)]
    if len(flat) == 0:
        return 'UNKNOWN'
    H, S, _ = np.median(flat, axis=0)
    if 10 <= H <= 60 and S > 40:
        return 'BEER'
    if (H < 10 or H > 160) and S > 60:
        return 'WINE'
    return 'UNKNOWN'


In [6]:
# --- 4) Helpers smoking
def quantize_frame(img, K=6):
    Z = img.reshape((-1,3)).astype(np.float32)
    _, labels, centers = cv2.kmeans(
        Z, K, None,
        (cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_MAX_ITER, 10, 1),
        5, cv2.KMEANS_RANDOM_CENTERS
    )
    return centers.astype(np.uint8), labels.flatten()

def label_color(bgr):
    hsv = cv2.cvtColor(np.uint8([[bgr]]), cv2.COLOR_BGR2HSV)[0,0]
    h, s, v = int(hsv[0]), int(hsv[1]), int(hsv[2])
    if (h < 30 or h > 150) and s > 100 and v > 150: return 'FLAME'
    if s < 50 and 50 < v < 200:                    return 'ASH'
    if 10 < h < 30 and 50 < s < 200 and 50 < v < 200: return 'TOBACCO'
    return 'OTHER'


In [11]:
# --- 5) Video loop
cap = cv2.VideoCapture(r"C:\Users\admin\OneDrive\Desktop\human-action-detection\livestream_dataset\Smoking\Smoking_v2.mp4")
frame_id = 0

In [12]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_id += 1
    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    detected_alcohol = False
    drink_rois = []

    # A) Detect containers & classify alcohol
    res = yolo_drink(rgb)
    for *b, conf, cls in res.xyxy[0]:
        nm = yolo_drink.names[int(cls)].lower()
        if nm in ('bottle', 'glass', 'cup', 'can'):
            x1, y1, x2, y2 = map(int, b)
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            seg = segment_liquid_region(crop)
            hsv = cv2.cvtColor(seg, cv2.COLOR_BGR2HSV)
            lbl = classify_liquid(hsv)
            if lbl in ('BEER', 'WINE'):
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(
                    frame, 'ALCOHOL DETECTED', (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2
                )
                logging.info(
                    f"Frame {frame_id}: Alcohol detected @({x1},{y1},{x2},{y2}) Label={lbl}"
                )
                detected_alcohol = True

    # B) Fallback to smoking detection
    if not detected_alcohol:
        face_res = mp_face.process(rgb)
        hand_res = mp_hands.process(rgb)
        mouth = None
        if face_res.multi_face_landmarks:
            lm = face_res.multi_face_landmarks[0].landmark
            mx = int((lm[13].x + lm[14].x)/2 * w)
            my = int((lm[13].y + lm[14].y)/2 * h)
            mouth = (mx, my)
        if hand_res.multi_hand_landmarks and mouth:
            for hl in hand_res.multi_hand_landmarks:
                pts = [(int(l.x*w), int(l.y*h)) for l in hl.landmark]
                thumb, idx, mid = pts[4], pts[8], pts[12]
                d_ti = math.hypot(thumb[0]-idx[0], thumb[1]-idx[1])
                d_im = math.hypot(idx[0]-mid[0], idx[1]-mid[1])
                palm = math.hypot(pts[0][0]-pts[9][0], pts[0][1]-pts[9][1])
                pinch_th, grasp_th = 0.3 * palm, 0.6 * palm
                if d_ti < pinch_th and d_im > pinch_th:
                    dist = math.hypot(idx[0]-mouth[0], idx[1]-mouth[1]) / w
                    if dist < SMOKE_DIST_THRESH:
                        cv2.putText(
                            frame, 'SMOKING', (idx[0], idx[1]-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 165, 255), 2
                        )
                        logging.info(
                            f"Frame {frame_id}: Smoking detected @({idx})"
                        )
                        break

    cv2.imshow("Combined Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Frame 33: Smoking detected @((391, 858))
Frame 34: Smoking detected @((389, 871))
Frame 35: Smoking detected @((385, 880))
Frame 36: Smoking detected @((381, 884))
Frame 37: Smoking detected @((375, 889))
Frame 38: Smoking detected @((372, 892))
Frame 39: Smoking detected @((372, 893))
Frame 40: Smoking detected @((370, 893))
Frame 41: Smoking detected @((371, 892))
Frame 42: Smoking detected @((373, 890))
Frame 43: Smoking detected @((378, 892))
Frame 44: Smoking detected @((381, 888))
